# Avaliação F2 pós-hoc: implantação simulada em 2018-08-01

Este notebook apresenta a avaliação F2, uma análise pós-hoc e exploratória que complementa a avaliação final F1 do notebook 09. As hipóteses testadas nasceram das observações feitas no período de teste do F1 (Dwork et al., 2015), e por isso o F2 não confirma nem refuta nada sobre o modelo pré-registrado. O resultado do F1 permanece como o resultado oficial, e o F2 serve apenas para formular hipóteses a serem testadas em dados posteriores, conforme a decisão P-06 de `docs/REGISTRO-DE-DECISOES.md` e o plano de `docs/PLANO-F2-POS-HOC.md`.

O notebook não seleciona um modelo para produção e não altera `production_eligible = false`. Todas as comparações são lidas como exploratórias, e nenhuma diferença é chamada de confirmada.

## 1. Natureza pós-hoc e verificação do regime da promessa

O F1 foi inconclusivo, e três sinais foram observados no seu período de teste: o prazo prometido, usado sozinho, foi o melhor preditor; o CatBoost superestimou o risco, porque a prevalência de violações caiu em relação ao treino; e as taxas históricas da rede mudaram de forma acentuada. As variantes do F2 foram desenhadas depois de essas observações serem conhecidas. Por isso, qualquer ganho em agosto de 2018 pode refletir, em parte, ajuste ao próprio período, e o desenho do F2 procura conter esse risco sem eliminá-lo.

A célula seguinte carrega os dados exatamente como o notebook 09 e verifica uma condição necessária da argumentação: o regime em que a promessa discrimina o atraso não existia no desenvolvimento. A tabela mostra, por mês de compra, a mediana do prazo prometido e, apenas para os meses anteriores a 2018-07-01, a prevalência e a AUC da regra do prazo prometido negativo. Para os meses seguintes, os rótulos não são lidos nesta verificação.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

LANG = "pt"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    F1_TRAIN_END,
    F2_EVAL_END,
    F2_EVAL_START,
    F2_UPDATE_DATE,
    build_modeling_frame,
    build_order_features,
    build_supervised_orders,
    describe_run,
    promise_regime_table,
    read_olist_tables,
    run_posthoc_evaluation,
)

POSTHOC_VERSION = "F2"
N_BOOT = 1000
SEED = 42
CACHE_ROOT = project_root / "outputs" / "runs"
F1_MODEL_PATH = project_root / "outputs" / "models" / "catboost_final" / "F1" / "model.joblib"

LABELS = {
    "pt": {
        "ideal": "calibração perfeita",
        "x": "probabilidade média prevista",
        "y": "taxa observada",
        "title": "Curva de confiabilidade em agosto de 2018 (10 bins)",
        "discarded": "Reamostragens descartadas por conter uma só classe: {n} de {total}",
        "post_hoc": "Execução pós-hoc (exploratória): {flag}",
        "eligible": "production_eligible = {value}",
        "reading": "Leitura de todas as comparações: {reading}",
        "update": "Atualização simulada em {date}; treino do F1 com compras anteriores a {train_end}",
        "window": "Avaliação: compras em [{start}, {end})",
        "available": "Pedidos supervisionados disponíveis: {n}",
    },
    "en": {
        "ideal": "perfect calibration",
        "x": "mean predicted probability",
        "y": "observed rate",
        "title": "Reliability curve in August 2018 (10 bins)",
        "discarded": "Resamples discarded for containing a single class: {n} of {total}",
        "post_hoc": "Post-hoc (exploratory) run: {flag}",
        "eligible": "production_eligible = {value}",
        "reading": "Reading of every comparison: {reading}",
        "update": "Simulated update on {date}; F1 training on purchases before {train_end}",
        "window": "Evaluation: purchases in [{start}, {end})",
        "available": "Supervised orders available: {n}",
    },
}[LANG]

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
print(LABELS["update"].format(date=F2_UPDATE_DATE, train_end=F1_TRAIN_END))
print(LABELS["window"].format(start=F2_EVAL_START, end=F2_EVAL_END))
print(LABELS["available"].format(n=len(modeling_frame)))

Atualização simulada em 2018-08-01; treino do F1 com compras anteriores a 2018-06-01
Avaliação: compras em [2018-08-01, 2018-09-01)
Pedidos supervisionados disponíveis: 96470


### Insight

A tabela de modelagem contém 96.470 pedidos supervisionados, e as janelas do F2 são aplicadas dentro de `run_posthoc_evaluation`. A data de entrega é usada somente para decidir quais rótulos já existiam na data de atualização e nunca entra como feature.

A tabela a seguir resume o comportamento da promessa mês a mês. Se a regra do prazo prometido negativo tivesse poder de discriminação no desenvolvimento, a AUC se afastaria de forma sistemática do acaso nos meses anteriores a julho de 2018. O gráfico a seguir mostra a mediana do prazo prometido em barras e a AUC da regra em linha, com uma referência para o acaso.

In [2]:
regime = promise_regime_table(modeling_frame)
display(regime)

from ecom_delivery_sla_viz import plot_promise_regime

plot_promise_regime(regime, language=LANG).show()

,purchase_month,orders,promise_median,labels_used,breach_prevalence,auc_negative_promise
0,2016-09,1,18.488449,True,1.000000,NaN
1,2016-10,265,54.152095,True,0.007547,0.714829
2,2016-12,1,26.030012,True,0.000000,NaN
3,2017-01,750,38.982905,True,0.029333,0.618007
4,2017-02,1653,30.176470,True,0.029643,0.614586
5,2017-03,2546,22.585220,True,0.045562,0.403697
6,2017-04,2303,25.470544,True,0.065567,0.570412
7,2017-05,3545,23.335255,True,0.029901,0.631309
8,2017-06,3135,23.419803,True,0.030303,0.527798
9,2017-07,3872,23.012668,True,0.027893,0.424702


### Insight

Nos meses de desenvolvimento de 2017-01 a 2018-06, a AUC da regra do prazo prometido negativo ficou entre 0,40 e 0,63, sem tendência, ou seja, próxima do acaso. A mediana do prazo prometido variou entre 21,3 e 27,4 dias de 2017-03 a 2018-06 e caiu para 19,3 dias em julho e 13,4 dias em agosto de 2018. O regime em que a promessa discrimina o atraso, portanto, não aparece nos rótulos de desenvolvimento, e nenhuma variante baseada nesse comportamento pode ser aprovada por eles. Essa é a razão pela qual o F2 é tratado como exploratório e pela qual o gate de desenvolvimento verifica apenas não inferioridade, e não ganho. No gráfico, os meses sem AUC não têm ponto na linha porque seus rótulos não são lidos nesta verificação.

## 2. Desenho da implantação simulada e disponibilidade de rótulos

A operação simulada coloca o modelo em uso em 1º de julho de 2018 e o atualiza em 2018-08-01, com os rótulos que já seriam conhecidos naquela data. Um rótulo é considerado conhecido quando o pedido foi entregue estritamente antes da data de atualização (Chapelle, 2014). A avaliação cobre as compras de agosto de 2018, e o intervalo de confiança é obtido por bootstrap em blocos diários da data de compra (Künsch, 1989), com 1.000 reamostragens e semente 42, de modo que todos os modelos são comparados nas mesmas reamostragens. Agosto tem poucas semanas, e blocos semanais produziriam intervalos sem sentido.

A ordem interna da execução é fixa: primeiro o gate de desenvolvimento das variantes C, D e E, sem nenhum contato com agosto, e depois a avaliação de agosto apenas das variantes aprovadas. A célula seguinte é a única chamada do projeto a `run_posthoc_evaluation` com `allow_test=True`. O resultado é congelado em `outputs/runs/final_evaluation/F2/` com a mesma semântica do F1: na primeira execução o resultado é calculado e uma linha é acrescentada ao ledger, e em qualquer execução posterior, inclusive no outro idioma, o resultado gravado é apenas carregado. Nenhum modelo novo é salvo em disco.

In [3]:
posthoc, posthoc_meta = run_posthoc_evaluation(
    modeling_frame,
    supervised_orders=supervised,
    allow_test=True,
    version=POSTHOC_VERSION,
    cache_root=CACHE_ROOT,
    f1_model_path=F1_MODEL_PATH,
    n_boot=N_BOOT,
    seed=SEED,
)
print(describe_run(posthoc_meta, language=LANG))
print(LABELS["post_hoc"].format(flag=posthoc_meta["post_hoc"]))

display(posthoc["provenance"])
ledger_path = CACHE_ROOT / "final_evaluation" / "LEDGER.jsonl"
if ledger_path.exists():
    display(pd.read_json(ledger_path, lines=True))

Resultados carregados de outputs/runs/final_evaluation/F2 (gerados em 2026-11-14 12:02, 16,3 min de treino; fingerprint a6fd373f)
Execução pós-hoc (exploratória): True


,version,post_hoc,reading,production_eligible,created_at,fingerprint,update_date,eval_start,eval_end_exclusive,known_label_orders,eval_orders,eval_blocks,gate_passed,gate_failed,n_boot,seed,f1_model_path,f1_model_sha256
0,F2,True,exploratório,False,2026-11-14T12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,2018-08-01,2018-08-01,2018-09-01,88097,6351,29,"C,E",D,1000,42,PROJECT_ROOT...,ace91068a027578a40abd579fa968938b332a74c5f484a...


,version,timestamp,fingerprint,train_orders,test_orders,post_hoc,n_boot,seed,gate_passed
0,F1,2026-11-14 10:53:17,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,False,1000,42,NaN
1,F2,2026-11-14 12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,88097,6351,True,1000,42,"C,E"


### Insight

O resultado foi calculado uma única vez, na execução em português, em 16,3 minutos, e gravado com o fingerprint `a6fd373f`; a execução em inglês apenas carregou o resultado gravado. O ledger passou a ter duas linhas: a do F1, inalterada, e a do F2, com a marcação `post_hoc` verdadeira e o registro das variantes aprovadas no gate, C e E. A procedência informa ainda 88.097 pedidos com rótulo conhecido na data de atualização e 6.351 pedidos avaliados em 29 blocos diários.

A tabela a seguir mostra, para as compras de junho e de julho de 2018, quantos pedidos tinham rótulo conhecido na data de atualização e qual era a prevalência de violações entre eles, em comparação com os pedidos ainda sem rótulo. O gráfico a seguir empilha os pedidos com rótulo conhecido e desconhecido em cada mês e traz a prevalência de cada grupo.

In [4]:
display(posthoc["label_availability"])

from ecom_delivery_sla_viz import plot_label_availability

plot_label_availability(posthoc["label_availability"], language=LANG).show()

,purchase_month,as_of,orders,known_labels,share_known,prevalence_known,unknown_labels,prevalence_unknown
0,2018-06,2018-08-01,6096,6070,0.995735,0.008402,26,0.769231
1,2018-07,2018-08-01,6156,4164,0.676413,0.003122,1992,0.097892


### Insight

Na data de atualização, 99,6% das compras de junho de 2018 já tinham rótulo conhecido, com prevalência de 0,84%, enquanto os 26 pedidos ainda não entregues tinham prevalência de 76,9%. Em julho, apenas 67,6% dos rótulos eram conhecidos, com prevalência de 0,31%, contra 9,79% entre os 1.992 pedidos ainda sem rótulo. Os pedidos mais lentos são justamente os que ainda não tinham sido entregues, de modo que os rótulos recentes conhecidos ficam enviesados em direção a entregas rápidas e a menos violações. Esse viés de rótulos lentos entra no treino de todas as variantes retreinadas e é mais intenso na janela recente usada na recalibração da variante D. A tabela é descritiva e não corrige o treino. O gráfico empilhado torna visível a parcela de rótulos ainda desconhecidos em cada mês.

## 3. Variantes A a E

A lista de variantes é fechada, todas são reportadas e nenhuma é escolhida pelo resultado de agosto. As baselines de prevalência, de prazo prometido negativo e de regressão logística com prazo e UF são reajustadas nos mesmos rótulos conhecidos da variante B.

| Código | Variante | Mecanismo testado |
|---|---|---|
| A | Modelo do F1 congelado, treinado com compras anteriores a 2018-06-01 | Referência sem atualização |
| B | Mesma configuração do F1, retreinada com todos os rótulos conhecidos na atualização | Retreino mensal |
| C | B com pesos de recência, com meia-vida de 90 dias | Adaptação a mudança de regime (Gama et al., 2014) |
| D | B com recalibração isotônica (Zadrozny e Elkan, 2002) ajustada somente nos rótulos entregues nos últimos 30 dias | Correção de mudança de prevalência |
| E | B com a feature `promise_vs_recent_median` | Captura de mudança na política de promessa |

A feature `promise_vs_recent_median` é a razão entre o prazo prometido do pedido e a mediana dos prazos prometidos nas compras dos 30 dias anteriores para a mesma UF de destino, calculada somente com compras estritamente anteriores, com recuo para a mediana global da mesma janela. Ela não usa rótulo e entra apenas na variante E.

As duas tabelas a seguir mostram o estado final de cada variante e o tamanho de cada conjunto utilizado, com as datas de compra e de entrega que o delimitam.

In [5]:
display(posthoc["variants"])
display(posthoc["populations"])

,code,variant,gated,scored,status
0,A,A_f1_frozen,False,True,avaliada
1,B,B_retrain,False,True,avaliada
2,C,C_recency,True,True,avaliada
3,D,D_recalibration,True,False,reprovada no gate
4,E,E_promise_feature,True,True,avaliada


,variant,role,orders,breaches,prevalence,first_purchase,last_purchase,first_delivery,last_delivery
0,A_f1_frozen,"train (F1, frozen)",77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,2016-10-11 13:46:32,2018-10-11 16:41:14
1,B_retrain,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
2,C_recency,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
3,E_promise_feature,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
4,baselines,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
5,all,evaluation (August purchases),6351,393,0.061880,2018-08-01 00:02:17,2018-08-29 15:00:37,2018-08-02 15:42:36,2018-09-25 00:47:25


### Insight

As variantes A, B, C e E foram pontuadas em agosto, e a variante D, reprovada no gate, não foi avaliada. As variantes retreinadas e as baselines foram ajustadas com 88.097 pedidos de rótulo conhecido, com prevalência de 6,72% e última entrega em 2018-07-31, contra 77.867 pedidos e prevalência de 7,53% no treino congelado do F1; a queda de prevalência reflete a entrada dos rótulos recentes de entregas rápidas. A população avaliada tem 6.351 compras de agosto, com 393 violações e prevalência de 6,19%. Nenhum rótulo com entrega a partir da data de atualização consta dos conjuntos de treino, e a variante A, sem retreino, serve de referência para medir o efeito da atualização.

## 4. Gate de desenvolvimento

Antes de qualquer contato com agosto, cada mecanismo das variantes C, D e E é aplicado nos folds de desenvolvimento, de 2017-10 a 2018-06 com embargo de 30 dias, e comparado com a configuração do F1 no mesmo fold. Nos folds, a data de atualização é o fim do treino. A idade usada nos pesos da variante C é relativa a essa data, e a janela de recalibração da variante D corresponde aos últimos 30 dias de compras do treino, porque os folds não usam a data de entrega como critério. Essa é uma adaptação do mecanismo real, que usa a data de entrega, e deve ser lida como tal.

Uma variante só segue para a avaliação de agosto se for não inferior no desenvolvimento: a AP média deve ser pelo menos 95% da AP média da configuração do F1, e o ECE médio deve ser no máximo 0,01 acima do ECE médio da configuração do F1. Variantes reprovadas aparecem na tabela final como reprovadas no gate e não são pontuadas em agosto.

A primeira tabela resume o gate por variante, com as médias, a razão de AP, a diferença de ECE e a decisão. A segunda traz as métricas por fold, incluindo a configuração do F1 usada como referência. O gráfico a seguir traz a AP de cada variante por fold, com a configuração do F1 tracejada.

In [6]:
display(posthoc["gate_summary"])
display(posthoc["gate_folds"])

from ecom_delivery_sla_viz import plot_metric_by_fold

plot_metric_by_fold(
    posthoc["gate_folds"],
    fold_col="fold",
    model_col="variant",
    metric_col="average_precision",
    reference="F1_config",
    language=LANG,
).show()

,variant,n_folds,mean_ap,reference_mean_ap,ap_ratio,ap_ratio_threshold,passes_ap,mean_ece,reference_mean_ece,ece_difference,ece_margin,passes_ece,passed,status
0,C_recency,9,0.170719,0.174999,0.975542,0.95,True,0.058680,0.057619,0.001061,0.01,True,True,avaliada
1,D_recalibration,9,0.145307,0.174999,0.830333,0.95,False,0.065182,0.057619,0.007563,0.01,True,False,reprovada no gate
2,E_promise_feature,9,0.169836,0.174999,0.970496,0.95,True,0.057674,0.057619,0.000056,0.01,True,True,avaliada


,fold,variant,update_date,n_validation,n_fit,n_calibration,average_precision,ece_10bins,brier
0,2017-10,F1_config,2017-09-01,4478,22264,0,0.084222,0.012413,0.039591
1,2017-10,C_recency,2017-09-01,4478,22264,0,0.078871,0.017704,0.039853
2,2017-10,D_recalibration,2017-09-01,4478,18232,4032,0.062233,0.013493,0.039909
3,2017-10,E_promise_feature,2017-09-01,4478,22264,0,0.075715,0.015709,0.039834
4,2017-11,F1_config,2017-10-02,7288,26538,0,0.207368,0.087310,0.113753
5,2017-11,C_recency,2017-10-02,7288,26538,0,0.209523,0.092434,0.114730
6,2017-11,D_recalibration,2017-10-02,7288,22417,4121,0.172526,0.081899,0.113618
7,2017-11,E_promise_feature,2017-10-02,7288,26538,0,0.186419,0.088827,0.114706
8,2017-12,F1_config,2017-11-01,5513,30892,0,0.122211,0.043552,0.069908
9,2017-12,C_recency,2017-11-01,5513,30892,0,0.120922,0.046552,0.070176


### Insight

As variantes C e E foram aprovadas, com razão de AP de 0,976 e 0,970 em relação à configuração do F1 e diferença de ECE de +0,001 e +0,0001. A variante D foi reprovada pela AP, com razão de 0,830 contra o limite de 0,95, embora a diferença de ECE, de +0,008, ficasse dentro da folga, e sua AP ficou abaixo da referência nos nove folds. O gate mede não inferioridade, e não ganho: C e E foram aprovadas por não prejudicar o desenvolvimento, e não por demonstrar benefício. Como nos folds a janela recente é definida pela data de compra, sem censura pela data de entrega, o gate não reproduz o viés de rótulos lentos observado na atualização real. A variante reprovada não é reavaliada com outros limiares depois de o resultado de agosto ser conhecido. O gráfico por fold permite comparar cada variante com a configuração do F1 em cada fold.

## 5. Avaliação em agosto com intervalos por blocos diários

As métricas de classificação são average precision (principal), ROC AUC, Brier, ECE em dez faixas e precisão e recall nos 5% e 10% de maior risco, todas com IC 95% do bootstrap por blocos diários. Para cada variante avaliada, a diferença pareada de AP é calculada contra a regressão logística com prazo e UF e contra a regra do prazo prometido negativo, com seu IC 95%. Nenhum rótulo de confirmado ou não confirmado é aplicado: a tabela traz apenas o sinal da diferença, se o intervalo exclui zero e a leitura fixa de exploratório. Brier e ECE não são definidos para a regra do prazo prometido negativo, que não é uma probabilidade.

A tabela a seguir traz a estimativa pontual na amostra completa e o intervalo de cada métrica, para as variantes avaliadas e para as baselines, e o estado de cada variante.

In [7]:
metrics = posthoc["classification_metrics"]
display(metrics[["model", "metric", "estimate", "ci_low", "ci_high", "status"]])
sampled = metrics.dropna(subset=["n_boot"])
print(LABELS["discarded"].format(n=int(sampled["n_discarded"].iloc[0]), total=int(sampled["n_boot"].iloc[0])))

,model,metric,estimate,ci_low,ci_high,status
0,A_f1_frozen,average_precision,0.069026,0.054170,0.083777,avaliada
1,A_f1_frozen,roc_auc,0.555992,0.516885,0.599745,avaliada
2,A_f1_frozen,brier,0.063841,0.048610,0.079037,avaliada
3,A_f1_frozen,ece_10bins,0.041298,0.034067,0.056659,avaliada
4,A_f1_frozen,precision_at_5pct,0.081761,0.053690,0.107878,avaliada
5,A_f1_frozen,recall_at_5pct,0.066158,0.042040,0.096222,avaliada
6,A_f1_frozen,precision_at_10pct,0.066038,0.046233,0.088289,avaliada
7,A_f1_frozen,recall_at_10pct,0.106870,0.073115,0.157310,avaliada
8,B_retrain,average_precision,0.068454,0.053698,0.082890,avaliada
9,B_retrain,roc_auc,0.556883,0.517592,0.600773,avaliada


Reamostragens descartadas por conter uma só classe: 0 de 1000


### Insight

Nenhuma das variantes avaliadas se destacou em agosto. A AP foi de 0,069 na variante A (IC 95% de 0,054 a 0,084), 0,068 em B, 0,067 em C e 0,069 em E, e a ROC AUC ficou entre 0,542 e 0,565, pouco acima da baseline de prevalência, cuja AP é 0,062. A regressão logística com prazo e UF alcançou AP de 0,094 (IC de 0,071 a 0,121) e AUC de 0,601, e a regra do prazo prometido negativo alcançou AP de 0,179 (IC de 0,105 a 0,284) e AUC de 0,777. A comparação entre A e B, que isola o efeito de atualizar o modelo, não mostra diferença relevante, e a feature de promessa da variante E praticamente não alterou o resultado de B. Nenhuma reamostragem foi descartada, e os intervalos refletem os 29 dias do período e a correlação dentro de cada dia.

A curva de confiabilidade compara, em dez faixas, a probabilidade média prevista com a taxa observada de violações para os modelos que produzem probabilidades. A tabela de precisão e recall nos 5% e 10% de maior risco resume o uso do score para priorização.

In [8]:
reliability = posthoc["reliability"]
display(reliability)

figure = go.Figure()
for model, part in reliability.groupby("model", sort=False):
    figure.add_trace(
        go.Scatter(x=part["mean_predicted"], y=part["observed_rate"], mode="lines+markers", name=model)
    )
figure.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line={"dash": "dash"}, name=LABELS["ideal"])
)
figure.update_layout(title=LABELS["title"], xaxis_title=LABELS["x"], yaxis_title=LABELS["y"])
figure.show()

display(posthoc["top_k"])

,model,bin,bin_lower,bin_upper,n,mean_predicted,observed_rate
0,A_f1_frozen,1,0.0,0.1,4521,0.053573,0.061712
1,A_f1_frozen,2,0.1,0.2,1213,0.139892,0.060181
2,A_f1_frozen,3,0.2,0.3,461,0.242664,0.060738
3,A_f1_frozen,4,0.3,0.4,123,0.339002,0.089431
4,A_f1_frozen,5,0.4,0.5,24,0.431044,0.083333
5,A_f1_frozen,6,0.5,0.6,2,0.558429,0.000000
6,A_f1_frozen,7,0.6,0.7,4,0.633337,0.000000
7,A_f1_frozen,8,0.7,0.8,2,0.718727,0.000000
8,A_f1_frozen,9,0.8,0.9,1,0.800651,0.000000
9,A_f1_frozen,10,0.9,1.0,0,NaN,NaN


,model,top_fraction,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high
0,A_f1_frozen,0.05,0.081761,0.053690,0.107878,0.066158,0.042040,0.096222
1,A_f1_frozen,0.10,0.066038,0.046233,0.088289,0.106870,0.073115,0.157310
2,B_retrain,0.05,0.075472,0.041973,0.101855,0.061069,0.033043,0.091252
3,B_retrain,0.10,0.064465,0.043829,0.088381,0.104326,0.073491,0.149655
4,C_recency,0.05,0.072327,0.040397,0.101984,0.058524,0.034085,0.085724
5,C_recency,0.10,0.069182,0.042699,0.092784,0.111959,0.072577,0.156874
6,E_promise_feature,0.05,0.075472,0.042207,0.103787,0.061069,0.032035,0.098046
7,E_promise_feature,0.10,0.067610,0.044630,0.089724,0.109415,0.070030,0.158787
8,baseline_prevalence,0.05,0.062893,0.010134,0.141540,0.050891,0.009091,0.111671
9,baseline_prevalence,0.10,0.056604,0.018122,0.117559,0.091603,0.032150,0.181626


### Insight

O erro de calibração mudou de direção com a atualização, sem diminuir. A variante A superestima o risco: nas faixas de 0,1 a 0,5, a probabilidade média prevista vai de 0,14 a 0,43, enquanto a taxa observada fica entre 0,06 e 0,09. As variantes retreinadas subestimam: B, C e E colocam, respectivamente, 6.302, 6.346 e 6.298 dos 6.351 pedidos na faixa abaixo de 0,1, com probabilidade média prevista de 0,021, 0,013 e 0,023, contra uma taxa observada de 0,062. O ECE ficou entre 0,039 e 0,049 nas quatro variantes, contra 0,005 da baseline de prevalência, cuja taxa de treino ficou próxima da de agosto. A subestimação dos modelos retreinados é compatível com o viés de rótulos lentos mostrado na seção 2, mas essa explicação não foi testada. A variante D, que atacaria esse desvio diretamente, não foi avaliada por ter sido reprovada no gate. Faixas com poucos pedidos devem ser lidas com cautela.

A tabela a seguir traz a diferença pareada de AP de cada variante contra as duas referências, com o IC 95% obtido nas mesmas reamostragens diárias. O gráfico de intervalos reúne a AP de cada variante e das baselines, com o IC 95%, e as diferenças pareadas contra as duas referências, com a linha de zero.

In [9]:
paired = posthoc["paired_difference"]
display(
    paired[
        [
            "model",
            "reference",
            "estimate_model",
            "estimate_reference",
            "difference",
            "ci_low",
            "ci_high",
            "difference_sign",
            "ci_excludes_zero",
            "reading",
            "status",
        ]
    ]
)

from ecom_delivery_sla_viz import plot_metric_intervals

plot_metric_intervals(metrics, metric="average_precision", paired=paired, language=LANG).show()

,model,reference,estimate_model,estimate_reference,difference,ci_low,ci_high,difference_sign,ci_excludes_zero,reading,status
0,A_f1_frozen,baseline_logistic_promise_plus_state,0.069026,0.093841,-0.024816,-0.043786,-0.010757,negative,True,exploratório,avaliada
1,A_f1_frozen,baseline_negative_promised_lead_days,0.069026,0.178748,-0.109723,-0.208806,-0.039556,negative,True,exploratório,avaliada
2,B_retrain,baseline_logistic_promise_plus_state,0.068454,0.093841,-0.025388,-0.044839,-0.009873,negative,True,exploratório,avaliada
3,B_retrain,baseline_negative_promised_lead_days,0.068454,0.178748,-0.110295,-0.209815,-0.040177,negative,True,exploratório,avaliada
4,C_recency,baseline_logistic_promise_plus_state,0.067454,0.093841,-0.026387,-0.046548,-0.011402,negative,True,exploratório,avaliada
5,C_recency,baseline_negative_promised_lead_days,0.067454,0.178748,-0.111294,-0.209972,-0.041853,negative,True,exploratório,avaliada
6,E_promise_feature,baseline_logistic_promise_plus_state,0.069296,0.093841,-0.024545,-0.043787,-0.009941,negative,True,exploratório,avaliada
7,E_promise_feature,baseline_negative_promised_lead_days,0.069296,0.178748,-0.109452,-0.209801,-0.039473,negative,True,exploratório,avaliada
8,D_recalibration,baseline_logistic_promise_plus_state,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate
9,D_recalibration,baseline_negative_promised_lead_days,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate


### Insight

Contra a regressão logística com prazo e UF, a diferença pareada de AP ficou entre −0,026 e −0,025 nas quatro variantes, com IC 95% inteiramente abaixo de zero (na variante A, de −0,044 a −0,011). Contra a regra do prazo prometido negativo, a diferença ficou em torno de −0,11, também com IC abaixo de zero (na variante A, de −0,209 a −0,040). Com várias variantes comparadas contra duas referências e sem ajuste para múltiplas comparações (Benjamini e Hochberg, 1995), essas diferenças são descritas como tamanho de efeito exploratório, e não como resultado confirmado. O sinal, ainda assim, é o mesmo em todas as comparações: nenhum dos mecanismos de atualização testados aproximou o CatBoost das referências baseadas na promessa. No gráfico, a linha de zero separa as diferenças a favor das diferenças contra cada referência.

## 6. Leitura exploratória e limitações

A leitura de todos os resultados deste notebook é exploratória. O F1 permanece como o resultado oficial, e o F2 é uma análise complementar. Uma variante só pode ser recomendada para um próximo ciclo, a ser validado em dados posteriores a agosto de 2018, e não para produção. A célula seguinte repete a leitura fixa e registra o campo `production_eligible`.

In [10]:
print(LABELS["reading"].format(reading=", ".join(sorted(paired["reading"].unique()))))
production_eligible = False
print(LABELS["eligible"].format(value=production_eligible))

Leitura de todas as comparações: exploratório
production_eligible = False


### Insight

A leitura exploratória do F2 é que nenhum dos mecanismos de atualização testados recuperou, em agosto de 2018, o desempenho das referências baseadas no prazo prometido. A explicação compatível com as tabelas, que não foi testada, combina três fatores: o regime em que a promessa discrimina o atraso não existe nos rótulos de treino, os rótulos recentes disponíveis na atualização são enviesados para entregas rápidas, e o gate de desenvolvimento não reproduz esse viés. A regra do prazo prometido negativo prevalece por ter sinal fixo e não depender de aprender essa relação. Duas hipóteses ficam registradas para um próximo ciclo, a validar em dados posteriores a agosto de 2018: treinar a atualização apenas com coortes de desfecho maduro, ou tratar os pedidos em trânsito como censurados (Kaplan e Meier, 1958), e usar a regra da promessa como alerta de mudança de política, e não como modelo.

Quatro limitações devem acompanhar qualquer citação do F2. Primeiro, agosto de 2018 já foi observado no F1, e as variantes foram desenhadas depois dessa observação, de modo que o período não é um holdout independente. Segundo, várias variantes são comparadas contra duas referências sem correção para múltiplas comparações. Terceiro, o período avaliado tem 29 blocos diários e uma população pequena, o que resulta em intervalos largos. Quarto, não existem dados posteriores a agosto de 2018 no conjunto, de modo que nenhuma hipótese pode ser testada fora do período já consultado; julho e agosto de 2018, consultados no F1 e no F2, ficam esgotados para novas avaliações.

Por essas razões, `production_eligible` permanece `false`, conforme a regra de governança do registro, e nenhuma variante é recomendada para uso operacional.

## Referências

Benjamini, Y. e Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society, Series B*, 57(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x

Chapelle, O. (2014). Modeling delayed feedback in display advertising. *Proceedings of the 20th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 1097–1105. https://doi.org/10.1145/2623330.2623634

Dwork, C., Feldman, V., Hardt, M., Pitassi, T., Reingold, O. e Roth, A. (2015). The reusable holdout: preserving validity in adaptive data analysis. *Science*, 349(6248), 636–638. https://doi.org/10.1126/science.aaa9375

Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M. e Bouchachia, A. (2014). A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), artigo 44. https://doi.org/10.1145/2523813

Kaplan, E. L. e Meier, P. (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association*, 53(282), 457–481. https://doi.org/10.1080/01621459.1958.10501452

Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *Annals of Statistics*, 17(3), 1217–1241. https://doi.org/10.1214/aos/1176347265

Zadrozny, B. e Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the 8th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694–699. https://doi.org/10.1145/775047.775151